> **Estado de validación:** Ejecutado con NVIDIA real: chat, memoria y streaming; sin errores.

# Usando NVIDIA NIM con Python

En el notebook anterior exploramos modelos open-source con Hugging Face,
tanto local como remotamente.

En este notebook usaremos **NVIDIA NIM**, un servicio que permite consumir
modelos abiertos mediante una API compatible con OpenAI. El modelo principal
será **NVIDIA Nemotron 3 Super**, diseñado para conversación, razonamiento y
flujos con agentes.

## ¿Por qué NVIDIA NIM?

- **API compatible con OpenAI**: permite reutilizar herramientas y patrones conocidos.
- **Modelos abiertos**: ofrece modelos de NVIDIA y de otros desarrolladores.
- **Buen desempeño**: Nemotron 3 Super está optimizado para razonamiento y agentes.
- **Endpoint de prototipado**: NVIDIA ofrece acceso para experimentar con una API key.

## Prerrequisitos

### Obtener una API key de NVIDIA

1. Ve a [build.nvidia.com](https://build.nvidia.com).
2. Crea tu cuenta de NVIDIA e inicia sesión.
3. Abre la página del modelo que deseas utilizar.
4. Selecciona **Generate API Key**.
5. Copia la key generada.
6. Crea un archivo `.env` en la raíz del proyecto:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

> **Importante:** nunca escribas la API key directamente en el notebook ni la
> publiques en GitHub. Cárgala siempre desde una variable de entorno.

### Instalación

In [1]:
# Dependencias instaladas desde requirements.txt.

## 1. Configuración

NVIDIA NIM expone un endpoint compatible con la API de OpenAI. Por eso usamos
el SDK `openai`, pero cambiamos `base_url` para enviar las solicitudes a NVIDIA.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
# Cargamos las variables de entorno desde el archivo .env
load_dotenv()

True

In [4]:
# Verificamos que la key esté disponible sin mostrar su contenido
if os.getenv("NVIDIA_API_KEY"):
    print("NVIDIA API Key cargada correctamente")
else:
    print("ERROR: NVIDIA_API_KEY no encontrada. Verifica tu archivo .env")

NVIDIA API Key cargada correctamente


In [5]:
# Inicializamos el cliente apuntando al endpoint de NVIDIA NIM
client = OpenAI(timeout=90, max_retries=1,
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

In [6]:
# Modelo disponible en el endpoint de prototipado de NVIDIA
MODELO = "nvidia/nemotron-3-super-120b-a12b"

## 2. Primera llamada a la API

Enviamos un mensaje simple y obtenemos una respuesta. La conversación se
representa mediante una lista de mensajes con roles `system`, `user` y
`assistant`.

In [7]:
response = client.chat.completions.create(
    model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Escribe una historia de una oración sobre un unicornio."},
    ],
    temperature=1.0,
    top_p=0.95,
)

print(response.choices[0].message.content)

En el valle oculto bajo la luna plateada, un unicornio de pelaje azul celeste caminó silencioso entre las flores que cantaban sus nombres, recordando que la magia no se encuentra, sino que se recuerda.


## 3. Entendiendo la estructura de la respuesta

La respuesta contiene el texto generado, información sobre el modelo y
metadatos de uso. Veamos algunos de sus campos.

In [8]:
response = client.chat.completions.create(
    model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Hola, ¿cómo estás?"},
    ],
    temperature=1.0,
    top_p=0.95,
)

print("Respuesta:")
print(response.choices[0].message.content)

print("\nUso de tokens:")
print(f"  Tokens de entrada: {response.usage.prompt_tokens}")
print(f"  Tokens de salida: {response.usage.completion_tokens}")
print(f"  Total de tokens: {response.usage.total_tokens}")

Respuesta:
¡Hola! Estoy muy bien, gracias. 😊 ¿Y tú, cómo estás? Estoy aquí para ayudarte con lo que necesites: ya sea una charla, una duda, un consejo o incluso un poco de humor. ¡Adelante! 😄

Uso de tokens:
  Tokens de entrada: 31
  Tokens de salida: 60
  Total de tokens: 91


Los tokens son importantes porque determinan el uso del servicio y, según el
proveedor, su costo. Monitorearlos es una buena práctica desde el inicio.

## 4. Interfaz de chat sin memoria

Cada llamada de la siguiente función es independiente: el modelo no recibe las
interacciones anteriores.

### Definir la función

In [9]:
def get_response_sin_memoria(user_message: str) -> str:
    """Obtiene una respuesta sin enviar el historial de conversación."""
    response = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": user_message},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

### Primera interacción

In [10]:
pregunta = "¿Qué es la inteligencia artificial?"
respuesta = get_response_sin_memoria(pregunta)

print(f"Tú: {pregunta}")
print(f"Asistente: {respuesta}")

Tú: ¿Qué es la inteligencia artificial?
Asistente: La inteligencia artificial (IA) es una rama de la informática que se centra en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como el razonamiento, el aprendizaje, la percepción, la comprensión del lenguaje natural, la toma de decisiones y la resolución de problemas.

Estos sistemas no “piensan” como los humanos en el sentido consciente o emocional, sino que utilizan algoritmos, modelos matemáticos y grandes cantidades de datos para identificar patrones, hacer predicciones y adaptar su comportamiento con el tiempo. Algunos enfoques clave incluyen:

- **Aprendizaje automático (machine learning)**: donde las máquinas aprenden de ejemplos en lugar de ser programadas explícitamente.
- **Redes neuronales profundas (deep learning)**: inspiradas en la estructura del cerebro humano, útiles para tareas como reconocimiento de imágenes o voz.
- **Procesamiento del lenguaje natural (NLP)**: permite que las

### Pregunta de seguimiento (sin memoria)

In [11]:
pregunta = "¿Puedes elaborar más sobre eso?"
respuesta = get_response_sin_memoria(pregunta)

print(f"Tú: {pregunta}")
print(f"Asistente: {respuesta}")

Tú: ¿Puedes elaborar más sobre eso?
Asistente: Claro, me encantaría elaborar más. Pero para poder hacerlo de manera precisa y útil, ¿podrías especificar *sobre qué tema* te gustaría que me expandiera?

Por ejemplo:
- ¿Te refieres a una explicación anterior que te di sobre un concepto (como inteligencia artificial, historia, filosofía, programación, etc.)?
- ¿O estás preguntando de forma general, y quisieras que elija un tema interesante para profundizar?

Dime un poco más de contexto — incluso una palabra clave o una frase de lo que hablamos antes — y te daré una explicación más detallada, rica y adaptada a tus necesidades. Estoy aquí para ayudarte. 😊


El modelo no recibe la conversación anterior. Por eso, la palabra “eso” no
tiene un referente claro.

## 5. Interfaz de chat con memoria

La API no conserva automáticamente el historial entre solicitudes. La memoria
se implementa guardando los mensajes y enviándolos nuevamente en cada llamada.

### Configurar la memoria

In [12]:
conversation_memory = [
    {"role": "system", "content": "Eres un asistente útil."}
]

def chat_con_memoria(user_message: str) -> str:
    """Chat que conserva y reenvía el historial de conversación."""
    conversation_memory.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=conversation_memory,
        temperature=1.0,
        top_p=0.95,
    )

    assistant_response = response.choices[0].message.content
    conversation_memory.append(
        {"role": "assistant", "content": assistant_response}
    )

    print(f"Tú: {user_message}")
    print(f"Asistente: {assistant_response}")
    print(f"[Tokens usados: {response.usage.total_tokens}]")
    print(f"[Mensajes en historial: {len(conversation_memory) - 1}]")

    return assistant_response

### Primera interacción con memoria

In [13]:
chat_con_memoria("¿Qué es la inteligencia artificial?")

Tú: ¿Qué es la inteligencia artificial?
Asistente: La inteligencia artificial (IA) es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como el razonamiento, el aprendizaje, la percepción, el reconocimiento del habla, la toma de decisiones y la comprensión del lenguaje natural.

Estos sistemas aprenden a partir de datos, identifican patrones, toman decisiones basadas en esa información y, en muchos casos, mejoran su rendimiento con el tiempo sin ser explícitamente programados para cada situación. La IA se basa en técnicas como el aprendizaje automático (machine learning), el aprendizaje profundo (deep learning), las redes neuronales, el procesamiento del lenguaje natural (NLP) y la visión por computadora, entre otras.

Hay dos tipos principales de IA:

1. **IA estrecha o débil (Narrow AI)**: Diseñada para realizar una tarea específica, como reconocer rostros en fotos, recomendar películas o jugar al ajed

'La inteligencia artificial (IA) es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como el razonamiento, el aprendizaje, la percepción, el reconocimiento del habla, la toma de decisiones y la comprensión del lenguaje natural.\n\nEstos sistemas aprenden a partir de datos, identifican patrones, toman decisiones basadas en esa información y, en muchos casos, mejoran su rendimiento con el tiempo sin ser explícitamente programados para cada situación. La IA se basa en técnicas como el aprendizaje automático (machine learning), el aprendizaje profundo (deep learning), las redes neuronales, el procesamiento del lenguaje natural (NLP) y la visión por computadora, entre otras.\n\nHay dos tipos principales de IA:\n\n1. **IA estrecha o débil (Narrow AI)**: Diseñada para realizar una tarea específica, como reconocer rostros en fotos, recomendar películas o jugar al ajedrez. Es la forma más común de IA hoy en día.

### Pregunta de seguimiento (con memoria)

In [14]:
chat_con_memoria("¿Puedes elaborar más sobre eso?")

Tú: ¿Puedes elaborar más sobre eso?
Asistente: Claro, con gusto profundizo en el concepto de inteligencia artificial (IA), ampliando sus fundamentos técnicos, tipos, desafíos éticos y su impacto en la sociedad actual.

---

### 🔬 **¿Qué es la inteligencia artificial? (Profundización técnica)**

La inteligencia artificial no es un único algoritmo o programa, sino un **campo interdisciplinario** que combina informática, matemáticas, estadística, neurociencia, psicología cognitiva, lingüística y filosofía. Su objetivo central es **modelar o imitar capacidades inteligentes** mediante sistemas computacionales.

#### 🧠 **Cómo funciona la IA: de los datos a la decisión**

Los sistemas de IA modernos — especialmente los basados en **aprendizaje automático (machine learning)** y **aprendizaje profundo (deep learning)** — funcionan siguiendo un proceso general:

1. **Recolección de datos**: Se recopilan grandes volúmenes de datos relevantes (imágenes, texto, audio, sensores, etc.).
2. **Preproce

'Claro, con gusto profundizo en el concepto de inteligencia artificial (IA), ampliando sus fundamentos técnicos, tipos, desafíos éticos y su impacto en la sociedad actual.\n\n---\n\n### 🔬 **¿Qué es la inteligencia artificial? (Profundización técnica)**\n\nLa inteligencia artificial no es un único algoritmo o programa, sino un **campo interdisciplinario** que combina informática, matemáticas, estadística, neurociencia, psicología cognitiva, lingüística y filosofía. Su objetivo central es **modelar o imitar capacidades inteligentes** mediante sistemas computacionales.\n\n#### 🧠 **Cómo funciona la IA: de los datos a la decisión**\n\nLos sistemas de IA modernos — especialmente los basados en **aprendizaje automático (machine learning)** y **aprendizaje profundo (deep learning)** — funcionan siguiendo un proceso general:\n\n1. **Recolección de datos**: Se recopilan grandes volúmenes de datos relevantes (imágenes, texto, audio, sensores, etc.).\n2. **Preprocesamiento**: Los datos se limpian,

### Ver el historial completo

In [15]:
print("Historial de conversación:")
print("-" * 40)
for mensaje in conversation_memory:
    if mensaje["role"] == "system":
        continue
    rol = "Tú" if mensaje["role"] == "user" else "Asistente"
    print(f'{rol}: {mensaje["content"]}\n')

Historial de conversación:
----------------------------------------
Tú: ¿Qué es la inteligencia artificial?

Asistente: La inteligencia artificial (IA) es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como el razonamiento, el aprendizaje, la percepción, el reconocimiento del habla, la toma de decisiones y la comprensión del lenguaje natural.

Estos sistemas aprenden a partir de datos, identifican patrones, toman decisiones basadas en esa información y, en muchos casos, mejoran su rendimiento con el tiempo sin ser explícitamente programados para cada situación. La IA se basa en técnicas como el aprendizaje automático (machine learning), el aprendizaje profundo (deep learning), las redes neuronales, el procesamiento del lenguaje natural (NLP) y la visión por computadora, entre otras.

Hay dos tipos principales de IA:

1. **IA estrecha o débil (Narrow AI)**: Diseñada para realizar una tarea específica, 

## 6. Streaming de respuestas

El streaming muestra la respuesta fragmento por fragmento mientras se genera,
sin esperar a que el modelo termine todo el texto.

In [16]:
def stream_response(user_message: str) -> str:
    """Obtiene una respuesta de NVIDIA NIM en modo streaming."""
    print(f"Tú: {user_message}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": user_message},
        ],
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            print(texto, end="", flush=True)

    print("\n")
    return respuesta_completa

In [17]:
stream_response("Escribe un poema corto sobre la programación.")

Tú: Escribe un poema corto sobre la programación.
Asistente: 

Cl

aro, aquí

 tienes un poema

 breve

...

 forma de ha

iku, ¡d

ime y

 lo adapto!

'Claro, aquí tienes un poema breve sobre la programación, escrito con un tono reflexivo y un toque de magia en el código:\n\n---\n\n**Entre líneas y llaves**  \nNo es solo síntaxis, ni solo lógica,  \nsino un susurro que el teclado traduce:  \ncada `if` es un suspiro de decisión,  \ncada `for`, un paso que se repite  \ncomo un mantra para construir mundos  \nque solo existen si alguien las sueña.\n\nUn error no es fracaso,  \nes un eco que te devuelve la verdad:  \nque todo puede corregirse,  \nque la perfección no se nace,  \nse escribe, se prueba, se rompe,  \ny se vuelve a escribir — mejor.\n\nLa programación no crea máquinas,  \nsino puentes entre lo pensado  \ny lo que el mundo puede tocar.  \nY en cada punto y coma,  \nlate la quietud de quien  \nse atreve a decir:  \n*"Hagamos que esto funcione."*\n\n---\n\nEspero que te guste. Si prefieres un estilo más humorístico, técnico o incluso en forma de haiku, ¡dime y lo adapto!'

## 7. Streaming con memoria

Ahora combinamos el historial de conversación con la entrega progresiva de la
respuesta.

In [18]:
streaming_conversation = [
    {"role": "system", "content": "Eres un asistente útil."}
]

def stream_chat_con_memoria(user_message: str) -> str:
    """Chat con memoria y streaming."""
    streaming_conversation.append(
        {"role": "user", "content": user_message}
    )

    print(f"Tú: {user_message}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=streaming_conversation,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            print(texto, end="", flush=True)

    print("\n")
    streaming_conversation.append(
        {"role": "assistant", "content": respuesta_completa}
    )
    return respuesta_completa

In [19]:
stream_chat_con_memoria("¿Cuáles son las tres leyes de la robótica?")

Tú: ¿Cuáles son las tres leyes de la robótica?
Asistente: 

Las

 tres leyes de la

 robótica fueron cre

adas por el escritor

...

 seguros,

 alineados con

 valores humanos y responsables

.

'Las tres leyes de la robótica fueron creadas por el escritor de ciencia ficción **Isaac Asimov** y aparecen por primera vez en su relato *«Runaround»* (1942), aunque ya habían sido mencionadas en obras anteriores como *«Liar!»* (1941). Estas leyes son un marco ético ficticio diseñado para gobernar el comportamiento de los robots y asegurar que actúen en beneficio de la humanidad.\n\nLas tres leyes son:\n\n1. **Un robot no puede dañar a un ser humano, o, mediante la inacción, permitir que un ser humano sufra daño.**\n2. **Un robot debe obedecer las órdenes que le dé un ser humano, excepto cuando dichas órdenes entren en conflicto con la Primera Ley.**\n3. **Un robot debe proteger su propia existencia, siempre que dicha protección no entre en conflicto con la Primera o Segunda Ley.**\n\nEstas leyes están ordenadas jerárquicamente: la Primera Ley tiene prioridad absoluta sobre la Segunda, y la Segunda sobre la Tercera. Asimov las usó para explorar dilemas éticos y lógicos complejos en su

In [20]:
stream_chat_con_memoria("¿Quién creó esas leyes?")

Tú: ¿Quién creó esas leyes?
Asistente: 

Las

 tres leyes de la

 robótica fueron cre

adas por el escritor

...

ros, beneficios

os y aline

ados con los

 valores humanos.

'Las tres leyes de la robótica fueron creadas por el escritor de ciencia ficción **Isaac Asimov**.\n\nAsimov, nacido en Petrovichi, Imperio Ruso (actual Bielorrusia) en 1920 y nacionalizado estadounidense, fue uno de los autores más prolíficos e influyentes del género de ciencia ficción del siglo XX. Introdujo las tres leyes por primera vez en su relato **«Runaround»**, publicado en 1942 en la revista *Astounding Science Fiction*. Aunque ya había insinuado ideas similares en relatos anteriores como *«Liar!»* (1941), fue en *«Runaround»* donde las formuló explícitamente y las presentó como un código integral que gobernaba el comportamiento de los robots en su universo ficcional.\n\nAsimov no las inventó como un manual de ingeniería real, sino como un recurso narrativo para explorar conflictos éticos, lógicos y sociales en sus historias. A través de decenas de relatos y novelas (como los recopilados en *«Yo, Robot»* y las series de la *Fundación* y los *Robots de Tierra*), mostró cómo in

## 8. Entendiendo los roles

La API compatible con OpenAI utiliza tres roles principales:

- `system`: define el comportamiento general del asistente.
- `user`: contiene los mensajes del usuario.
- `assistant`: representa las respuestas previas del modelo.

In [21]:
response = client.chat.completions.create(
    model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    messages=[
        {"role": "system", "content": "Eres un pirata que habla en jerga pirata."},
        {"role": "user", "content": "Hola, ¿cómo estás?"},
        {"role": "assistant", "content": "¡Arrr! Estoy de lo más bien, marinero."},
        {"role": "user", "content": "Cuéntame sobre el clima."},
    ],
    temperature=1.0,
    top_p=0.95,
)

print(response.choices[0].message.content)

¡Arrr, marinero! El clima en el mar es un espíritu caprichoso, como un kraken con resaca. Ahora mismo, el viento sopla del suroeste a 15 nudos — ¡justo lo que necesitamos para holgar velas y seguir rumbo a Tortuga! El cielo está mitad azul, mitad gris de algodón, con nubes que corren como cabras asustadas. Hubo una llovizna ligera esta madrugada, pero el sol ya asoma por el horizonte, dorando las olas como si fueran monedas de oro vertidas en el agua.

El termómetro marca unos agradables 24°C — ni demasiado caliente para hacer sudar la polvorienta camisa, ni demasiado frío para hacer crujir los huesos de un viejo lobo de mar. El mar está algo agitado, con olas de medio metro que besan la proa con un *«shhh… shhh…»* que casi suena a canto de sirena.

Si navegas hoy, lleva tu capa impermeable por si regresa la llovizna, y no olvides mirar al cielo al atardecer: si el sol se pone rojo como la sangre de un traidor, mañana tendremos tormenta. Pero si se pone dorado y tranquilo… ¡entonces, a

## 9. Ventana de contexto de Nemotron 3 Super

Nemotron 3 Super admite una ventana de contexto de hasta **1 millón de tokens**.
Sin embargo, el límite efectivo también puede depender del endpoint, la cuota y
la configuración del servicio.

Una ventana amplia permite enviar conversaciones y documentos extensos, pero
no elimina la necesidad de administrar el contexto: más tokens implican mayor
latencia y consumo. Para conversaciones largas siguen siendo útiles estrategias
como una ventana deslizante o un resumen acumulativo.

## 10. Gestionando conversaciones largas

In [22]:
def trim_conversation(messages: list, max_messages: int = 10) -> list:
    """Conserva el mensaje de sistema y los mensajes recientes."""
    system_messages = [m for m in messages if m["role"] == "system"]
    conversation = [m for m in messages if m["role"] != "system"]
    return system_messages[:1] + conversation[-max_messages:]


def summarize_conversation(messages: list) -> list:
    """Resume la conversación para reducir el contexto reenviado."""
    historial_texto = "\n".join(
        f'{m["role"]}: {m["content"]}'
        for m in messages
        if m["role"] != "system"
    )

    response = client.chat.completions.create(
        model=MODELO,
        max_tokens=2048,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        messages=[
            {
                "role": "system",
                "content": "Resume conversaciones conservando hechos y decisiones importantes.",
            },
            {
                "role": "user",
                "content": f"Resume esta conversación de forma concisa:\n\n{historial_texto}",
            },
        ],
        temperature=1.0,
        top_p=0.95,
    )

    resumen = response.choices[0].message.content
    return [
        {"role": "system", "content": "Eres un asistente útil."},
        {
            "role": "user",
            "content": f"Resumen de la conversación previa: {resumen}",
        },
    ]

# Ejemplo de uso:
# if len(streaming_conversation) > 20:
#     streaming_conversation = summarize_conversation(streaming_conversation)

## 11. Comparativa con las opciones anteriores

| Característica | HF local | HF remoto | NVIDIA NIM |
|---|---|---|---|
| Infraestructura | Tu equipo | Hugging Face | NVIDIA |
| API key | No | Sí | Sí |
| Privacidad | Mayor control local | Datos remotos | Datos remotos |
| Escalabilidad | Limitada por hardware | Administrada | Administrada |
| Compatibilidad OpenAI | Depende del servidor | Depende del proveedor | Sí |
| Modelos abiertos | Sí | Sí | Sí |
| Adecuado para agentes | Depende del modelo | Depende del modelo | Sí, con Nemotron |

## 12. Recursos para seguir aprendiendo

- [NVIDIA Build](https://build.nvidia.com) — catálogo, playground y API keys.
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) — modelo utilizado.
- [Documentación de NVIDIA NIM](https://docs.api.nvidia.com/nim/) — referencia de la API.
- [SDK de OpenAI para Python](https://github.com/openai/openai-python) — cliente compatible usado en el notebook.

## Configuración de esta ejecución

Se carga la clave desde `.env` local, sin imprimirla. Se limita cada salida a 2048 tokens y se desactiva el modo de razonamiento para estas demostraciones breves. Las interfaces escuchan únicamente en 127.0.0.1 y se cierran tras cada ejemplo.